# Matkakuluväiteanalyysi

Tämä muistikirja näyttää, kuinka luodaan agenteja, jotka käyttävät liitännäisiä käsitelläkseen matkakuluja paikallisista kuittikuvista, laativat matkakuluvaatimuksen sähköpostin ja visualisoivat kulutietoja piirakkakaavion avulla. Agentit valitsevat toiminnot dynaamisesti tehtävän kontekstin mukaan.

Vaiheet:
1. OCR-agentti käsittelee paikallisen kuittikuvan ja poimii matkakulutiedot.
2. Sähköpostiantti laatii matkakuluväitteestä sähköpostin.

### Esimerkki matkakulu-tilanteesta:
Kuvittele, että olet työntekijä ja matkustat työpalaveriin toiseen kaupunkiin. Yritykselläsi on käytäntö korvata kaikki kohtuulliset matkaan liittyvät kulut. Tässä erittely mahdollisista matkakuluista:
- Liikenne:
Lentoliput meno-paluu-matkalle kotikaupungista kohdekaupunkiin.
Taksi- tai kyytipalvelut lentokentälle ja sieltä pois.
Paikallinen liikkuminen kohdekaupungissa (esim. julkinen liikenne, vuokra-autot tai taksit).

- Majoitus:
Kolmen yön hotelli majoitus keskitason liikehotellissa lähellä tapaamispaikkaa.

- Ruokailu:
Päivittäinen ruokailukorvaus aamiaiselle, lounaalle ja päivälliselle yrityksen päivärahan mukaan.

- Muut kulut:
Pysäköintimaksut lentokentällä.
Internet-yhteysmaksut hotellissa.
Tippi tai pienet palvelumaksut.

- Asiakirjat:
Toimitat kaikki kuitit (lennot, taksit, hotelli, ruoat jne.) ja täytetyn kuluraportin korvausta varten.


## Tuo tarvittavat kirjastot

Tuo tarvittavat kirjastot ja moduulit muistiinpanoille.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Määrittele kulumallit

 Luo Pydantic-malli yksittäisille kuluillesi ja ExpenseFormatter-luokka, joka muuntaa käyttäjän kyselyn rakenteelliseksi kulutiedoksi.

 Jokainen kulu esitetään muodossa:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Työkalujen määrittely - Sähköpostin luominen

Luo työkalutoiminto kulukorvaushakemuksen lähettämiseksi sähköpostilla.
- Tämä työkalu käyttää Microsoft Agent Frameworkin `@tool`-koristetta.
- Se laskee kulujen kokonaissumman ja muotoilee tiedot sähköpostin rungoksi.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Työkalu matka-avustusten purkamiseen kuittikuvista

Luo työkalutoiminto matka-avustusten purkamiseen kuittikuvista.
- Tämä työkalu käyttää Microsoft Agent Frameworkin `@tool`-koristetta.
- Se lukee kuittikuvan, koodaa sen base64-muotoon ja palauttaa datan URI-muodossa agentin analysoitavaksi.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Kustannusten käsittely

Määrittele agentit ja yhdistä ne peräkkäiseen työnkulkuun käyttämällä `WorkflowBuilder`-luokkaa.
- OCR-agentti poimii rakenteelliset kustannustiedot kuittikuvasta käyttäen `load_receipt_image`-työkalua.
- Sähköpostiapuri ottaa poimitut tiedot ja luo ammattimaisen kustannuskorvauspyyntöviestin käyttäen `generate_expense_email`-työkalua.
- `WorkflowBuilder` ja `add_edge` muodostavat peräkkäisen putken: OCR-agentti → Sähköpostiapuri.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Päätoiminto

Kokoa peräkkäinen työnkulku ja suorita se käsittelemään kuittikuva sekä luomaan kulukorvausviesti.


> **Huom:** Tämä työnkulku siirtää tällä hetkellä kuittikuvan base64-tekstinä, jota useimmat chat-mallit (mukaan lukien gpt-4o) eivät käsittele kuvana.
> Se saattaa myös ylittää mallin kontekstikehyksen. Suositeltavaa on suorittaa OCR Azure AI Visionilla (tai muulla OCR-työkalulla) ja siirtää ainoastaan poimittu teksti, tai muokata työnkulku siten, että kuva lähetetään `image_url`-viestinä.
> Jos haluat vain välttää kontekstivirheitä, kokeile pienempää kuittikuvaa tai mallia, jolla on suurempi kontekstikehys.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
